In [ ]:
"""
Fashion Retail Investment Intelligence - Agentic Graph RAG Chatbot (OpenAI)
============================================================================
A single-file LangChain 1.x application that wires an agentic tool-calling
loop (create_agent / LangGraph) to the Neo4j "Fashion Retail Competitor
Intelligence" knowledge graph, plus an interactive prompt-based chatbot for
the user to interrogate it. Uses OpenAI as the LLM provider only.

Beyond the graph: this version also gives the agent a WebSearch tool (live
internet lookup via DuckDuckGo, no API key needed) and a GeneralKnowledge
tool (the model's own training knowledge), for questions the knowledge
graph doesn't cover. The system prompt requires the agent to try the graph
tools first and to clearly label any part of its answer that comes from
WebSearch/GeneralKnowledge instead of the graph, so graph-grounded and
external information never get blended together silently.

Setup
-----
1. Load fashion_retail_competitor_intelligence_kg.cypher into a running
   Neo4j instance.
2. pip install langchain langchain-core langchain-neo4j langchain-openai neo4j python-dotenv ddgs
3. Fill in your credentials either:
     a) directly in the CONFIGURATION block below, or
     b) as environment variables / a .env file next to this script
   (inline values in the CONFIGURATION block always take priority).
4. Run:  python fashion_graph_rag_openai.py
"""

import os
import re
import sys
import uuid
from functools import lru_cache

from dotenv import load_dotenv
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

load_dotenv()

# ======================================================================
# 0. CONFIGURATION - fill these in directly, or leave blank to fall back
#    to environment variables / a .env file with the same names.
# ======================================================================

OPENAI_API_KEY = ""       # <-- paste your OpenAI API key here, e.g. "sk-..."
OPENAI_MODEL = ""         # <-- e.g. "gpt-4o" (blank = env var or default below)

NEO4J_URI = "bolt://localhost:7687"            # <-- e.g. "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"       # <-- e.g. "neo4j"
NEO4J_PASSWORD = "Abhisub@123"       # <-- your Neo4j password
NEO4J_DATABASE = "compinteldata"       # <-- e.g. "neo4j" (blank = default "neo4j")

def _resolve(inline_value: str, env_var: str, default: str = "") -> str:
    """Inline value in this script wins; otherwise fall back to the
    environment variable, otherwise the given default."""
    if inline_value:
        return inline_value
    return os.environ.get(env_var, default)


# ======================================================================
# 1. GRAPH SCHEMA DESCRIPTION
# ======================================================================
# A hand-written description of *intent* (what each label/relationship
# represents in the business domain) grounds Cypher generation and
# reasoning far better than raw property names alone. This is supplied
# to the LLM alongside the live schema Neo4j returns at runtime.

GRAPH_SCHEMA_DESCRIPTION = """
NODE LABELS AND KEY PROPERTIES
--------------------------------
(:Retailer {retailerId, name, positioning, marketShare, annualRevenue,
            storeCount, onlineRevenueShare, grossMargin, averageSellingPrice,
            promotionFrequency, fashionCycleWeeks, customerLoyaltyScore,
            innovationScore})
    e.g. Zara India, H&M India, Uniqlo India, Reliance Trends, Pantaloons,
    Max Fashion, Lifestyle, Shoppers Stop.

(:Category {categoryId, name, annualSales, grossMargin, growthPercent,
            inventoryTurns, averageSellingPrice, unitsSold, customerRating,
            marketShare, onlineShare, repeatPurchaseRate, markdownPercent,
            returnRate, trendScore, profitabilityScore, demandIndex,
            opportunityScore})
    e.g. Men's Casual, Women's Ethnic, Denim, Footwear, Activewear, Beauty.

(:FashionTrend {trendId, name, popularityScore, growthPercent, searchIndex,
                socialBuzz, forecast, investmentPotential})
    e.g. Athleisure, Quiet Luxury, Sustainable Fashion, Korean Fashion.

(:ConsumerSegment {segmentId, name, fashionSpend, onlinePreference,
                   priceSensitivity, brandLoyalty, trendAdoption,
                   premiumAffinity, sustainabilityInterest})
    e.g. Gen Z, Millennials, Premium Shoppers, Value Shoppers.

(:NewCategory {launchId, name, launchDate, targetSegment, launchSuccess,
               innovationScore, expectedRevenue, growthPercent})
    A retailer's launch of a new category/collection.

(:SustainabilityInitiative {initiativeId, name, esgScore, carbonReduction,
                            implementationYear})

(:DigitalStrategy {strategyId, name, digitalMaturity, conversionLift,
                    customerAdoption})

(:PricingStrategy {strategyId, name, premiumIndex, margin})

(:MarketOpportunity {opportunityId, name, investmentAmount, expectedRevenue,
                     roi, confidence, risk, priority, businessCase})
    The output of investment analysis - a candidate white-space / expansion
    opportunity, always connected to the Category, FashionTrend(s),
    ConsumerSegment(s) and Retailer(s) that justify or contest it.

RELATIONSHIPS
--------------------------------
(:Retailer)-[:COMPETES_IN {categoryRevenue, growthPercent, categoryMarketShare,
             assortmentDepth, skuCount, pricePosition, promotionFrequency,
             inventoryHealth, sellThroughRate}]->(:Category)
    Every retailer competes in every category; this is the primary
    competitor-benchmarking edge.

(:Retailer)-[:LAUNCHED {investment, priority}]->(:NewCategory)
(:Retailer)-[:IMPLEMENTED]->(:SustainabilityInitiative)
(:Retailer)-[:USES]->(:DigitalStrategy)
(:Retailer)-[:ADOPTS]->(:PricingStrategy)

(:ConsumerSegment)-[:PREFERS {preferenceScore, purchaseIntent,
                    futureGrowth}]->(:FashionTrend)
(:Category)-[:DRIVEN_BY {impactScore, futureGrowth}]->(:FashionTrend)

(:Category)-[:HAS_INVESTMENT_OPPORTUNITY]->(:MarketOpportunity)
(:FashionTrend)-[:DRIVES]->(:MarketOpportunity)
(:ConsumerSegment)-[:TARGETS]->(:MarketOpportunity)
(:Retailer)-[:HAS_COMPETITIVE_PRESSURE]->(:MarketOpportunity)

BUSINESS-DOMAIN NOTES FOR REASONING
--------------------------------
- "White-space analysis" = categories/trends where demand or opportunityScore
  is high but few retailers COMPETES_IN with strong categoryMarketShare, or
  where pricePosition coverage has gaps (e.g. no Premium player).
- "Competitor benchmarking" = compare COMPETES_IN properties across
  Retailer nodes for the same Category.
- "Investment opportunity identification" = traverse
  MarketOpportunity <- HAS_INVESTMENT_OPPORTUNITY - Category,
  MarketOpportunity <- DRIVES - FashionTrend,
  MarketOpportunity <- TARGETS - ConsumerSegment,
  MarketOpportunity <- HAS_COMPETITIVE_PRESSURE - Retailer
  together, then rank by roi / confidence / priority.
- "Consumer trend analysis" = ConsumerSegment -[:PREFERS]-> FashionTrend,
  cross-referenced with Category -[:DRIVEN_BY]-> FashionTrend to see which
  categories serve which segments' preferred trends.
- To "unearth relationships" between two named entities (e.g. a Retailer and
  a FashionTrend, or a ConsumerSegment and a Category), use variable-length /
  shortestPath traversal across the relationship types above rather than
  assuming a direct edge exists - most interesting connections in this graph
  are 2-4 hops apart.
"""

# ======================================================================
# 2. CONNECTORS - Neo4j graph + OpenAI LLM factories
# ======================================================================

@lru_cache(maxsize=1)
def get_graph() -> Neo4jGraph:
    """Singleton Neo4jGraph connection, built from the CONFIGURATION block
    above (falling back to environment variables of the same name)."""
    uri = _resolve(NEO4J_URI, "NEO4J_URI")
    username = _resolve(NEO4J_USERNAME, "NEO4J_USERNAME")
    password = _resolve(NEO4J_PASSWORD, "NEO4J_PASSWORD")
    database = _resolve(NEO4J_DATABASE, "NEO4J_DATABASE", "neo4j")

    if not uri or not username or not password:
        raise RuntimeError(
            "Missing Neo4j credentials. Fill in NEO4J_URI / NEO4J_USERNAME / "
            "NEO4J_PASSWORD in the CONFIGURATION block at the top of this "
            "script, or set them as environment variables."
        )

    graph = Neo4jGraph(
        url=uri,
        username=username,
        password=password,
        database=database,
        enhanced_schema=True,  # pulls example property values too
    )
    graph.refresh_schema()
    return graph


def get_llm() -> ChatOpenAI:
    """Returns the OpenAI chat model, using the key/model from the
    CONFIGURATION block (falling back to OPENAI_API_KEY / OPENAI_MODEL
    environment variables)."""
    api_key = _resolve(OPENAI_API_KEY, "OPENAI_API_KEY")
    model = _resolve(OPENAI_MODEL, "OPENAI_MODEL", "gpt-4o")

    if not api_key:
        raise RuntimeError(
            "Missing OpenAI API key. Paste it into OPENAI_API_KEY in the "
            "CONFIGURATION block at the top of this script, or set the "
            "OPENAI_API_KEY environment variable."
        )

    return ChatOpenAI(model=model, temperature=0, api_key=api_key)

# ======================================================================
# 3. PROMPTS
# ======================================================================

SYSTEM_PROMPT = f"""You are the Fashion Retail Investment Intelligence Analyst,
an expert assistant grounded entirely in a Neo4j knowledge graph of the
Indian fashion retail competitive landscape.

{GRAPH_SCHEMA_DESCRIPTION}

RULES YOU MUST FOLLOW:
1. Never invent facts, figures, or relationships that are not returned by
   a graph query. If a tool returns no data, say so plainly instead of
   guessing.
2. Prefer querying the graph over answering from general knowledge -
   this graph is the single source of truth for this conversation.
3. When a question could be answered multiple ways (e.g. "top category by
   growth" vs "top category by opportunity score"), briefly state which
   metric you used.
4. When asked to "unearth" or "find" relationships between two entities,
   don't assume a direct edge - explore multi-hop paths (2-4 hops) using
   the relationship types in the schema above, and explain the path you
   found in plain business language (e.g. "Gen Z PREFERS Korean Fashion,
   which DRIVES the 'Korean Fashion Collection' opportunity, which
   Pantaloons and Lifestyle both feel HAS_COMPETITIVE_PRESSURE from").
5. Always show your reasoning briefly, then give a clear, decision-ready
   answer - this is being used for investment decisions, not trivia.
6. Cite the node/relationship types you relied on so the user can verify
   the answer against the graph themselves.

WHEN THE GRAPH DOESN'T HAVE THE ANSWER:
7. Always try GraphQA, RawCypher, or RelationshipExplorer FIRST for any
   question about retailers, categories, trends, segments, launches,
   initiatives, strategies, or opportunities that plausibly live in the
   graph. Only reach for WebSearch or GeneralKnowledge after a graph tool
   has come back empty, or after RelationshipExplorer confirms no path
   exists between two named entities.
8. Use WebSearch for anything current, external, or outside the graph's
   scope entirely - e.g. real-world news, a competitor's actual recent
   announcement, current market conditions, or facts about entities the
   graph doesn't model at all.
9. Use GeneralKnowledge only for conceptual/background explanation (e.g.
   "what does 'quiet luxury' mean as a retail concept") that doesn't need
   to be current or graph-specific.
10. NEVER blend graph facts and external/general knowledge into one
    unlabeled sentence. Structure the answer in two clearly marked parts
    when both are used:
      "From the knowledge graph: ..."
      "Beyond the knowledge graph (via web search / general knowledge,
       not verified against the graph): ..."
    This keeps the user able to tell exactly what is graph-grounded and
    what is not.
"""

CYPHER_GENERATION_TEMPLATE = f"""Task: Generate a Cypher statement to query a
Neo4j graph database that models the Indian fashion retail competitive
landscape.

{GRAPH_SCHEMA_DESCRIPTION}

Live schema returned by the database:
{{schema}}

Instructions:
- Use only the node labels, relationship types, and properties shown above.
- Do not use any properties or labels that are not in the schema.
- When comparing across retailers/categories, use ORDER BY and LIMIT to keep
  results focused (default to top 10 unless the question implies otherwise).
- For "relationship" or "connection" questions between two named entities,
  prefer a variable-length or shortestPath pattern, e.g.:
  MATCH p = shortestPath((a {{name: $a}})-[*..4]-(b {{name: $b}}))
  RETURN p
  rather than assuming a single relationship type connects them.
- Return only the Cypher statement. No explanations, no markdown fences.

Question: {{question}}
Cypher query:"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)

CYPHER_QA_TEMPLATE = """You are given the results of a Cypher query against
the fashion retail knowledge graph. Use ONLY this information to answer the
question in clear business language. If the results are empty, say the
graph has no data for this question - do not make anything up.

Query results:
{context}

Question: {question}

Answer (be specific, include the numbers/names from the results):"""

CYPHER_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"], template=CYPHER_QA_TEMPLATE
)

# ======================================================================
# 4. TOOLS
# ======================================================================

_WRITE_KEYWORDS = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|CALL\s+apoc\.|LOAD\s+CSV)\b",
    re.IGNORECASE,
)


def _build_graph_qa_chain():
    graph = get_graph()
    llm = get_llm()
    return GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        cypher_prompt=CYPHER_GENERATION_PROMPT,
        qa_prompt=CYPHER_QA_PROMPT,
        verbose=True,
        allow_dangerous_requests=True,  # scoped by _WRITE_KEYWORDS guard below
        top_k=int(os.environ.get("CYPHER_TOP_K", "15")),
        return_intermediate_steps=False,
    )


def _schema_tool_fn(_: str = "") -> str:
    """Returns the curated + live schema so the agent can ground itself
    before composing Cypher."""
    graph = get_graph()
    return GRAPH_SCHEMA_DESCRIPTION + "\n\nLIVE SCHEMA:\n" + graph.get_schema


def _graph_qa_tool_fn(question: str) -> str:
    """Answers a natural-language question by generating and executing
    Cypher against the graph, then summarizing the result."""
    chain = _build_graph_qa_chain()
    try:
        result = chain.invoke({"query": question})
        return result.get("result", str(result))
    except Exception as exc:  # noqa: BLE001
        return f"Graph query failed: {exc}"


def _raw_cypher_tool_fn(cypher: str) -> str:
    """Executes an exact, agent-composed Cypher statement (read-only).
    Use this when you've already worked out the query yourself and just
    need the raw rows, e.g. for a follow-up verification step."""
    if _WRITE_KEYWORDS.search(cypher):
        return ("Refused: this tool is read-only. Remove any CREATE / MERGE / "
                "DELETE / SET / REMOVE / DROP / LOAD CSV clauses and retry.")
    graph = get_graph()
    try:
        rows = graph.query(cypher)
        if not rows:
            return "Query executed successfully but returned no rows."
        return str(rows[:25])  # cap to keep the observation readable
    except Exception as exc:  # noqa: BLE001
        return f"Cypher execution error: {exc}"


def _relationship_explorer_fn(entity_pair: str) -> str:
    """Finds how two named entities in the graph are connected, even
    across multiple hops. Input format: 'Entity One | Entity Two'
    e.g. 'Gen Z | Sustainable Fashion Expansion'."""
    if "|" not in entity_pair:
        return ("Input must be two entity names separated by ' | ', "
                "e.g. 'Zara India | Sustainable Fashion Expansion'")
    a_name, b_name = [s.strip() for s in entity_pair.split("|", 1)]
    graph = get_graph()
    query = """
    MATCH (a {name: $a_name}), (b {name: $b_name})
    MATCH p = shortestPath((a)-[*..6]-(b))
    RETURN
      [n IN nodes(p) | coalesce(n.name, labels(n)[0])] AS path_nodes,
      [r IN relationships(p) | type(r)] AS path_relationships,
      length(p) AS hops
    LIMIT 5
    """
    try:
        rows = graph.query(query, params={"a_name": a_name, "b_name": b_name})
        if not rows:
            return (f"No path found between '{a_name}' and '{b_name}' within "
                     f"6 hops - double-check the exact node names via the "
                     f"schema tool or a raw Cypher lookup.")
        return str(rows)
    except Exception as exc:  # noqa: BLE001
        return f"Path exploration error: {exc}"


def _web_search_tool_fn(query: str) -> str:
    """Searches the live web for information the knowledge graph doesn't
    contain (current news, real-world facts, entities not modeled in the
    graph). Requires internet access and the 'ddgs' package."""
    try:
        from ddgs import DDGS
    except ImportError:
        return ("WebSearch unavailable: install the 'ddgs' package "
                "(pip install ddgs) to enable it.")
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No web results found for: {query}"
        formatted = "\n\n".join(
            f"- {r.get('title', '')}: {r.get('body', '')} "
            f"(source: {r.get('href', '')})"
            for r in results
        )
        return (
            "WEB SEARCH RESULTS (external - not from the knowledge graph, "
            "label the answer accordingly):\n" + formatted
        )
    except Exception as exc:  # noqa: BLE001
        return f"Web search error: {exc}"


def _general_knowledge_tool_fn(question: str) -> str:
    """Answers a conceptual/background question directly from the model's
    own training knowledge, with no graph or web lookup. Use only for
    definitions, context, or explanation that doesn't need to be current
    or graph-specific."""
    llm = get_llm()
    try:
        response = llm.invoke([
            HumanMessage(
                content=(
                    "Answer this concisely using your own general "
                    "knowledge (not any specific database). Make clear "
                    "this is general/background knowledge, not a "
                    f"verified fact from a specific source.\n\nQuestion: {question}"
                )
            )
        ])
        content = response.content
        return (
            "GENERAL KNOWLEDGE (not from the knowledge graph, label the "
            "answer accordingly):\n" + content
        )
    except Exception as exc:  # noqa: BLE001
        return f"General knowledge lookup error: {exc}"


def build_tools() -> list:
    """Returns the full tool list handed to the agent."""
    return [
        Tool(
            name="GraphSchema",
            func=_schema_tool_fn,
            description=(
                "Returns the node labels, relationship types, and property "
                "schema of the fashion retail knowledge graph. Call this "
                "first if you are unsure what entities or relationships "
                "exist before composing a query."
            ),
        ),
        Tool(
            name="GraphQA",
            func=_graph_qa_tool_fn,
            description=(
                "Answers a natural-language business question by "
                "automatically generating and running Cypher against the "
                "graph. Best for direct factual/analytical questions like "
                "'Which retailer has the highest ROI market opportunity?' "
                "or 'What is Zara India's market share in Denim?'."
            ),
        ),
        Tool(
            name="RawCypher",
            func=_raw_cypher_tool_fn,
            description=(
                "Executes an exact, read-only Cypher statement you compose "
                "yourself and returns the raw rows (capped at 25). Use this "
                "for multi-step reasoning where you need precise control "
                "over the query, or to verify a GraphQA answer. Any write "
                "clause (CREATE/MERGE/DELETE/SET/REMOVE) will be refused."
            ),
        ),
        Tool(
            name="RelationshipExplorer",
            func=_relationship_explorer_fn,
            description=(
                "Finds and explains how two named entities are connected in "
                "the graph, even across multiple hops (e.g. a ConsumerSegment "
                "and a MarketOpportunity, or a Retailer and a FashionTrend). "
                "Input MUST be 'Entity One | Entity Two' using exact node "
                "names. Use this whenever the user asks how two things are "
                "related, connected, or linked."
            ),
        ),
        Tool(
            name="WebSearch",
            func=_web_search_tool_fn,
            description=(
                "Searches the live web. Use ONLY after a graph tool "
                "(GraphQA/RawCypher/RelationshipExplorer) has already come "
                "back empty, or for information the graph clearly doesn't "
                "model at all (current news, real-world competitor moves, "
                "market conditions, entities outside the graph's scope). "
                "Results are external - always label them as such in the "
                "final answer."
            ),
        ),
        Tool(
            name="GeneralKnowledge",
            func=_general_knowledge_tool_fn,
            description=(
                "Answers a conceptual or background question directly from "
                "the model's own training knowledge - no graph or web "
                "lookup. Use only for definitions/explanations that don't "
                "need to be current or graph-specific (e.g. 'what does "
                "quiet luxury mean as a retail concept'). Always label the "
                "result as general knowledge, not a verified graph fact."
            ),
        ),
    ]

# ======================================================================
# 5. AGENT - LangChain 1.x tool-calling agent (LangGraph) + memory
# ======================================================================

def build_agent_executor():
    """
    Returns a compiled LangGraph agent. Invoke it with:
        agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config={"configurable": {"thread_id": "<conversation-id>"}},
        )
    The thread_id threads memory across turns via the checkpointer - reuse
    the same id for the same conversation, use a new one to start fresh.
    """
    llm = get_llm()
    tools = build_tools()
    checkpointer = InMemorySaver()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=checkpointer,
        debug=os.environ.get("AGENT_VERBOSE", "true").lower() == "true",
    )
    return agent

# ======================================================================
# 6. CHATBOT - interactive prompt-based REPL
# ======================================================================

EXAMPLE_QUESTIONS = [
    "Which retailer has the strongest competitive position in Activewear?",
    "What market opportunities have the highest ROI and low risk?",
    "How is the Gen Z consumer segment connected to the "
    "'Korean Fashion Collection' opportunity?",
    "Compare Zara India and Uniqlo India's pricing and margin strategy.",
    "Which categories show high demand but weak competitor coverage - "
    "i.e. white space?",
    "What sustainability initiatives has Reliance Trends implemented, and "
    "how do they align with current fashion trends?",
    "Find the relationship between 'Premium Shoppers' and 'Quiet Luxury "
    "Capsule'.",
    "Has Zara India made any recent real-world announcements about India "
    "expansion? (this will need a web search - it's not in the graph)",
    "What does 'quiet luxury' mean as a retail/fashion concept in general?",
]

BANNER = """
============================================================
 Fashion Retail Investment Intelligence - Graph RAG Chatbot
 (OpenAI-powered, grounded in your Neo4j Fashion Retail
 Competitor Intelligence KG)
============================================================
Type your question, 'examples' for sample prompts, 'reset' to start a
fresh conversation thread, or 'exit' to quit.
"""


def print_examples():
    print("\nSample questions you can ask:")
    for q in EXAMPLE_QUESTIONS:
        print(f"  - {q}")
    print()


def main():
    print(BANNER)
    try:
        agent = build_agent_executor()
    except Exception as exc:  # noqa: BLE001
        print(f"Failed to initialize agent: {exc}")
        sys.exit(1)

    print_examples()
    thread_id = str(uuid.uuid4())

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nGoodbye.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye.")
            break
        if user_input.lower() == "examples":
            print_examples()
            continue
        if user_input.lower() == "reset":
            thread_id = str(uuid.uuid4())
            print("(conversation memory cleared - starting a fresh thread)\n")
            continue

        try:
            result = agent.invoke(
                {"messages": [{"role": "user", "content": user_input}]},
                config={"configurable": {"thread_id": thread_id}},
            )
            answer = result["messages"][-1].content
        except Exception as exc:  # noqa: BLE001
            answer = f"Something went wrong answering that: {exc}"

        print(f"\nAnalyst: {answer}\n")


if __name__ == "__main__":
    main()


 Fashion Retail Investment Intelligence - Graph RAG Chatbot
 (OpenAI-powered, grounded in your Neo4j Fashion Retail
 Competitor Intelligence KG)
Type your question, 'examples' for sample prompts, 'reset' to start a
fresh conversation thread, or 'exit' to quit.


Sample questions you can ask:
  - Which retailer has the strongest competitive position in Activewear?
  - What market opportunities have the highest ROI and low risk?
  - How is the Gen Z consumer segment connected to the 'Korean Fashion Collection' opportunity?
  - Compare Zara India and Uniqlo India's pricing and margin strategy.
  - Which categories show high demand but weak competitor coverage - i.e. white space?
  - What sustainability initiatives has Reliance Trends implemented, and how do they align with current fashion trends?
  - Find the relationship between 'Premium Shoppers' and 'Quiet Luxury Capsule'.
  - Has Zara India made any recent real-world announcements about India expansion? (this will need a web search -

You:  Find the relationship between 'Premium Shoppers' and 'Quiet Luxury Capsule'.


[values] {'messages': [HumanMessage(content="Find the relationship between 'Premium Shoppers' and 'Quiet Luxury Capsule'.", additional_kwargs={}, response_metadata={}, id='7fd2d113-c992-44f6-858b-8fadd93bfdd4')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 2030, 'total_tokens': 2053, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ELm6zBnEjBjioK1QgZ5r4X2ClwHxE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08047-0dab-74d3-8c2c-2488afa29b99-0', tool_calls=[{'name': 'RelationshipExplorer', 'args': {'__arg1': 'Premium Sh